<a href="https://colab.research.google.com/github/gabiuxo/Algoritmos-de-Aprendizaje-Automatico/blob/main/Retos_SVM_Gabriel_Elizondo_Martinez.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Retos del Estudiante - Diagnóstico y Evaluación de SVM

**Nombre:** Gabriel Elizondo Martinez  
**Matrícula:** AL07009102  
**Tema:** Máquinas de Vectores de Soporte (SVM)

En esta actividad voy a completar los cuatro niveles y el desafío final. Para tomar las decisiones voy a usar lo visto sobre kernels, `C`, `gamma`, validación cruzada, diagnóstico visual y matriz de confusión.

## Librerías

Primero importo todo lo que necesito. Voy a mantener `StandardScaler` dentro de los pipelines para que el escalado se aprenda solamente con los datos de entrenamiento y no exista fuga de información.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import (
    make_moons,
    make_classification,
    load_breast_cancer,
    load_wine,
)
from sklearn.model_selection import (
    train_test_split,
    StratifiedKFold,
    GridSearchCV,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    ConfusionMatrixDisplay,
    classification_report,
    make_scorer,
)

sns.set_theme(style="whitegrid")
pd.set_option("display.max_colwidth", None)

print("Librerías importadas correctamente.")

# Nivel 1 - Clasificar fronteras

En este primer reto voy a comparar una SVM lineal contra una SVM con kernel RBF sobre datos con forma de medias lunas. Antes de ejecutar el código espero que el modelo lineal tenga problemas, porque una recta no puede seguir bien la forma curva de los datos.

In [ ]:
# generar el dataset del reto
X_1, y_1 = make_moons(n_samples=250, noise=0.18, random_state=42)

models_1 = [
    (
        "Lineal, C=0.5",
        Pipeline([
            ("scaler", StandardScaler()),
            ("svc", SVC(kernel="linear", C=0.5)),
        ]),
    ),
    (
        "RBF, C=10, gamma=1",
        Pipeline([
            ("scaler", StandardScaler()),
            ("svc", SVC(kernel="rbf", C=10, gamma=1)),
        ]),
    ),
]

# crear la malla donde se evaluará la función de decisión
xx_1, yy_1 = np.meshgrid(
    np.linspace(X_1[:, 0].min() - 0.5, X_1[:, 0].max() + 0.5, 300),
    np.linspace(X_1[:, 1].min() - 0.5, X_1[:, 1].max() + 0.5, 300),
)
grid_1 = np.c_[xx_1.ravel(), yy_1.ravel()]

results_1 = []
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for ax, (title, model) in zip(axes, models_1):
    model.fit(X_1, y_1)
    prediction = model.predict(X_1)
    decision = model.decision_function(grid_1).reshape(xx_1.shape)

    results_1.append({
        "Modelo": title,
        "Accuracy de entrenamiento": accuracy_score(y_1, prediction),
        "Vectores de soporte": int(model.named_steps["svc"].n_support_.sum()),
    })

    ax.contourf(xx_1, yy_1, decision > 0, alpha=0.25, cmap="coolwarm")
    ax.contour(
        xx_1,
        yy_1,
        decision,
        levels=[-1, 0, 1],
        linestyles=["--", "-", "--"],
        colors="black",
    )
    ax.scatter(X_1[:, 0], X_1[:, 1], c=y_1, cmap="coolwarm", edgecolor="black", s=42)
    ax.set_title(title)

plt.tight_layout()
plt.show()

display(pd.DataFrame(results_1).round(4))

### Mi diagnóstico

- **Modelo lineal:** lo clasifico como **subajuste**. La frontera es una recta que atraviesa las dos medias lunas y deja varios puntos de ambos colores del lado equivocado. Su accuracy de entrenamiento fue **0.8520**, lo cual confirma que no tiene suficiente capacidad para representar la forma de los datos.
- **Modelo RBF:** lo clasifico como un **balance adecuado**. La frontera se curva siguiendo la separación natural entre las lunas, pero no crea pequeñas islas alrededor de puntos individuales. Su accuracy de entrenamiento fue **0.9840**.

La comparación me mostró que una frontera más compleja sí se justifica cuando la geometría real es no lineal.

# Nivel 2 - Diagnosticar un dataset nuevo

Ahora voy a crear un dataset con ruido, separar entrenamiento y prueba, y comparar tres configuraciones. Voy a revisar tanto la forma de las fronteras como la diferencia entre las métricas de entrenamiento y prueba.

In [ ]:
# crear el dataset nuevo
X_2, y_2 = make_classification(
    n_samples=300,
    n_features=2,
    n_redundant=0,
    n_clusters_per_class=1,
    class_sep=0.7,
    flip_y=0.08,
    random_state=15,
)

# reservar 25% para prueba
X_train_2, X_test_2, y_train_2, y_test_2 = train_test_split(
    X_2,
    y_2,
    test_size=0.25,
    stratify=y_2,
    random_state=42,
)

models_2 = {
    "Lineal C=0.1": Pipeline([
        ("scaler", StandardScaler()),
        ("svc", SVC(kernel="linear", C=0.1)),
    ]),
    "RBF C=1, gamma=scale": Pipeline([
        ("scaler", StandardScaler()),
        ("svc", SVC(kernel="rbf", C=1, gamma="scale")),
    ]),
    "RBF C=100, gamma=10": Pipeline([
        ("scaler", StandardScaler()),
        ("svc", SVC(kernel="rbf", C=100, gamma=10)),
    ]),
}

# malla para las tres fronteras
xx_2, yy_2 = np.meshgrid(
    np.linspace(X_2[:, 0].min() - 0.7, X_2[:, 0].max() + 0.7, 350),
    np.linspace(X_2[:, 1].min() - 0.7, X_2[:, 1].max() + 0.7, 350),
)
grid_2 = np.c_[xx_2.ravel(), yy_2.ravel()]

results_2 = []
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, (name, model) in zip(axes, models_2.items()):
    model.fit(X_train_2, y_train_2)
    train_prediction = model.predict(X_train_2)
    test_prediction = model.predict(X_test_2)
    decision = model.decision_function(grid_2).reshape(xx_2.shape)

    results_2.append({
        "Modelo": name,
        "Accuracy train": accuracy_score(y_train_2, train_prediction),
        "Accuracy test": accuracy_score(y_test_2, test_prediction),
        "Precision test": precision_score(y_test_2, test_prediction),
        "Recall test": recall_score(y_test_2, test_prediction),
        "F1 test": f1_score(y_test_2, test_prediction),
    })

    ax.contourf(xx_2, yy_2, decision > 0, alpha=0.25, cmap="coolwarm")
    ax.contour(
        xx_2,
        yy_2,
        decision,
        levels=[-1, 0, 1],
        linestyles=["--", "-", "--"],
        colors="black",
    )
    ax.scatter(X_2[:, 0], X_2[:, 1], c=y_2, cmap="coolwarm", edgecolor="black", s=35)
    ax.set_title(name)

plt.tight_layout()
plt.show()

results_2_df = pd.DataFrame(results_2).set_index("Modelo")
display(results_2_df.round(4))

### Mi diagnóstico

- **Lineal, C=0.1:** tiene una frontera simple y no intenta perseguir el ruido. Visualmente puede considerarse de baja capacidad porque la recta atraviesa la zona donde se mezclan las clases. Aun así, obtuvo el mejor accuracy de prueba, **0.7733**, por lo que en este dataset una frontera sencilla funciona razonablemente bien.
- **RBF, C=1, gamma=scale:** genera una curva suave y no crea regiones alrededor de cada punto. Lo considero **balanceado**, aunque su accuracy de prueba fue **0.7600**, así que la complejidad adicional no produjo una mejora frente al modelo lineal.
- **RBF, C=100, gamma=10:** muestra **sobreajuste**. La frontera es muy ondulada y forma pequeñas islas para capturar puntos individuales. Su accuracy subió a **0.9467 en entrenamiento**, pero bajó a **0.6800 en prueba**.

El tercer modelo es el ejemplo más claro de que una frontera que se adapta mucho al entrenamiento no necesariamente generaliza mejor.

# Nivel 3 - Reporte de clasificación completo

En este reto voy a trabajar con Breast Cancer Wisconsin. El dataset usa `0 = malignant` y `1 = benign`. Voy a entrenar exactamente la SVM RBF solicitada y después revisar la matriz de confusión y el reporte por clase.

In [ ]:
# cargar Breast Cancer Wisconsin
cancer_3 = load_breast_cancer()
X_3, y_3 = cancer_3.data, cancer_3.target

X_train_3, X_test_3, y_train_3, y_test_3 = train_test_split(
    X_3,
    y_3,
    test_size=0.20,
    stratify=y_3,
    random_state=42,
)

model_3 = Pipeline([
    ("scaler", StandardScaler()),
    ("svc", SVC(kernel="rbf", C=1.0, gamma="scale")),
])

model_3.fit(X_train_3, y_train_3)
y_pred_3 = model_3.predict(X_test_3)

cm_3 = confusion_matrix(y_test_3, y_pred_3)
malignant_as_benign = int(cm_3[0, 1])
benign_as_malignant = int(cm_3[1, 0])

print(f"Accuracy: {accuracy_score(y_test_3, y_pred_3):.4f}")
print(f"Tumores malignos clasificados como benignos: {malignant_as_benign}")
print(f"Tumores benignos clasificados como malignos: {benign_as_malignant}")
print()
print("Reporte de clasificación:")
print(classification_report(
    y_test_3,
    y_pred_3,
    target_names=cancer_3.target_names,
    digits=4,
))

ConfusionMatrixDisplay(
    confusion_matrix=cm_3,
    display_labels=cancer_3.target_names,
).plot(cmap="Blues", colorbar=False)
plt.title("Matriz de confusión - SVM RBF")
plt.show()

### Mi análisis de los errores

El modelo obtuvo **0.9825 de accuracy**. La matriz muestra **1 tumor maligno clasificado como benigno** y **1 tumor benigno clasificado como maligno**.

El primer error es el más crítico porque podría retrasar estudios o tratamiento para una persona que sí tiene un tumor maligno. Por eso no me quedaría únicamente con accuracy. También revisaría especialmente el **recall de la clase malignant**, que fue **0.9762**, y su F1-score. En este problema es preferible aceptar algunas alertas falsas antes que dejar pasar casos malignos reales.

# Nivel 4 - Flujo completo con Wine

Voy a convertir `load_wine` en un problema binario: `1` significa que el vino pertenece a la clase original 0 y `0` significa que pertenece a cualquiera de las otras clases. Después voy a optimizar una SVM RBF y compararla con un baseline lineal bajo la misma división.

In [ ]:
# cargar y convertir Wine a clasificación binaria
wine_4 = load_wine()
X_4, y_original_4 = wine_4.data, wine_4.target
y_4 = (y_original_4 == 0).astype(int)

X_train_4, X_test_4, y_train_4, y_test_4 = train_test_split(
    X_4,
    y_4,
    test_size=0.20,
    stratify=y_4,
    random_state=42,
)

cv_4 = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

rbf_pipeline_4 = Pipeline([
    ("scaler", StandardScaler()),
    ("svc", SVC(kernel="rbf")),
])

param_grid_4 = {
    "svc__C": [0.1, 1, 10, 100],
    "svc__gamma": [0.001, 0.01, 0.1, 1],
}

search_4 = GridSearchCV(
    rbf_pipeline_4,
    param_grid_4,
    scoring="f1",
    cv=cv_4,
    n_jobs=-1,
)
search_4.fit(X_train_4, y_train_4)
y_pred_rbf_4 = search_4.predict(X_test_4)

# baseline lineal con el mismo protocolo
baseline_4 = Pipeline([
    ("scaler", StandardScaler()),
    ("svc", SVC(kernel="linear", C=1.0)),
])
baseline_4.fit(X_train_4, y_train_4)
y_pred_linear_4 = baseline_4.predict(X_test_4)

def binary_metrics(y_true, y_pred):
    return {
        "Accuracy": accuracy_score(y_true, y_pred),
        "Precision": precision_score(y_true, y_pred),
        "Recall": recall_score(y_true, y_pred),
        "F1-score": f1_score(y_true, y_pred),
    }

comparison_4 = pd.DataFrame({
    "RBF optimizado": binary_metrics(y_test_4, y_pred_rbf_4),
    "Lineal baseline": binary_metrics(y_test_4, y_pred_linear_4),
}).T

print("Mejores hiperparámetros:", search_4.best_params_)
print(f"Mejor F1 promedio en validación cruzada: {search_4.best_score_:.4f}")
display(comparison_4.round(4))

print("Reporte del modelo RBF optimizado:")
print(classification_report(
    y_test_4,
    y_pred_rbf_4,
    target_names=["Otras clases", "Clase original 0"],
    digits=4,
))

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, title, prediction in [
    (axes[0], "RBF optimizado", y_pred_rbf_4),
    (axes[1], "Baseline lineal", y_pred_linear_4),
]:
    ConfusionMatrixDisplay(
        confusion_matrix=confusion_matrix(y_test_4, prediction),
        display_labels=["Otras", "Clase 0"],
    ).plot(ax=ax, cmap="Blues", colorbar=False)
    ax.set_title(title)

plt.tight_layout()
plt.show()

### Mi comparación

GridSearchCV eligió `C=100` y `gamma=0.01`, con un F1 promedio de validación cruzada de **1.0000**. En prueba, tanto el RBF optimizado como el baseline lineal clasificaron correctamente los **36 casos** y obtuvieron **1.0000** en todas las métricas.

Aunque RBF ganó la búsqueda, en esta partición no superó al modelo lineal. Por eso no puedo justificar su complejidad adicional solamente con estos resultados. Elegiría el baseline lineal por ser más sencillo y confirmaría el empate con otras particiones, porque el conjunto de prueba es pequeño.

# Desafío final - Defender un clasificador SVM

Para cerrar voy a construir un flujo completo con Breast Cancer Wisconsin. Voy a comparar kernels lineal y RBF mediante validación cruzada y reservar el conjunto de prueba para una sola evaluación final.

Como la clase `malignant` está codificada con `0`, usaré un scorer que indique explícitamente `pos_label=0`. Así la búsqueda optimiza el F1 de los tumores malignos y no el de los benignos por accidente.

In [ ]:
# 1. partición de datos
cancer_final = load_breast_cancer()
X_final, y_final = cancer_final.data, cancer_final.target

X_train_final, X_test_final, y_train_final, y_test_final = train_test_split(
    X_final,
    y_final,
    test_size=0.20,
    stratify=y_final,
    random_state=42,
)

print("Dimensiones:")
print("X_train:", X_train_final.shape)
print("X_test:", X_test_final.shape)
print("y_train:", y_train_final.shape)
print("y_test:", y_test_final.shape)

# 2. escalado seguro dentro del pipeline
pipeline_final = Pipeline([
    ("scaler", StandardScaler()),
    ("svc", SVC()),
])

# 3. búsqueda de dos kernels
param_grid_final = [
    {
        "svc__kernel": ["linear"],
        "svc__C": [0.01, 0.1, 1, 10, 100],
    },
    {
        "svc__kernel": ["rbf"],
        "svc__C": [0.1, 1, 10, 100],
        "svc__gamma": [0.001, 0.01, 0.1, 1],
    },
]

cv_final = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
malignant_f1 = make_scorer(f1_score, pos_label=0)

search_final = GridSearchCV(
    pipeline_final,
    param_grid_final,
    scoring=malignant_f1,
    cv=cv_final,
    n_jobs=-1,
    return_train_score=True,
)
search_final.fit(X_train_final, y_train_final)

print()
print("Configuración ganadora:", search_final.best_params_)
print(f"F1 maligno promedio en CV: {search_final.best_score_:.4f}")

# revisar las mejores configuraciones de la búsqueda
cv_results_final = pd.DataFrame(search_final.cv_results_)
top_results_final = cv_results_final.sort_values("rank_test_score")[
    ["params", "mean_test_score", "std_test_score", "mean_train_score", "rank_test_score"]
].head(6)
display(top_results_final.round(4))

In [ ]:
# 4. evaluación final en datos no vistos
best_model_final = search_final.best_estimator_
y_pred_final = best_model_final.predict(X_test_final)

# baseline lineal bajo la misma división
baseline_final = Pipeline([
    ("scaler", StandardScaler()),
    ("svc", SVC(kernel="linear", C=1.0)),
])
baseline_final.fit(X_train_final, y_train_final)
y_pred_baseline_final = baseline_final.predict(X_test_final)

def cancer_metrics(y_true, y_pred):
    return {
        "Accuracy": accuracy_score(y_true, y_pred),
        "Precision malignant": precision_score(y_true, y_pred, pos_label=0),
        "Recall malignant": recall_score(y_true, y_pred, pos_label=0),
        "F1 malignant": f1_score(y_true, y_pred, pos_label=0),
        "Errores totales": int((y_true != y_pred).sum()),
    }

comparison_final = pd.DataFrame({
    "SVM optimizada": cancer_metrics(y_test_final, y_pred_final),
    "Baseline lineal": cancer_metrics(y_test_final, y_pred_baseline_final),
}).T

display(comparison_final.round(4))

print("Reporte del modelo ganador:")
print(classification_report(
    y_test_final,
    y_pred_final,
    target_names=cancer_final.target_names,
    digits=4,
))

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, title, prediction in [
    (axes[0], "SVM optimizada", y_pred_final),
    (axes[1], "Baseline lineal", y_pred_baseline_final),
]:
    ConfusionMatrixDisplay(
        confusion_matrix=confusion_matrix(y_test_final, prediction),
        display_labels=cancer_final.target_names,
    ).plot(ax=ax, cmap="Blues", colorbar=False)
    ax.set_title(title)

plt.tight_layout()
plt.show()

## Defensa final del modelo

### 1. Partición

Separé los 569 registros en **455 casos de entrenamiento** y **114 de prueba**, usando estratificación. El conjunto de prueba no participó en la selección de hiperparámetros.

### 2. Preprocesamiento

El `StandardScaler` quedó dentro del `Pipeline`. Esto significa que en cada fold el escalador aprende únicamente de la parte de entrenamiento correspondiente y evita fuga de información.

### 3. Modelo seleccionado

GridSearchCV comparó kernels lineal y RBF. La configuración ganadora fue **RBF con `C=10` y `gamma=0.01`**, con un F1 promedio de validación cruzada de **0.9668 para la clase malignant**.

### 4. Evaluación

La SVM optimizada obtuvo **0.9825 de accuracy**, **0.9762 de precision**, **0.9762 de recall** y **0.9762 de F1** para tumores malignos. Cometió dos errores: un maligno fue clasificado como benigno y un benigno como maligno.

No dibujé una frontera de decisión 2D porque el dataset tiene 30 características. Reducirlas a dos solo para dibujar una frontera cambiaría el problema y podría dar una imagen engañosa. En su lugar utilicé matrices de confusión como evidencia visual directa de los errores.

### 5. Comparación y conclusión

El baseline lineal obtuvo **0.9737 de accuracy** y **0.9647 de F1 maligno**, con tres errores. El RBF optimizado redujo los errores de tres a dos y mejoró el F1 maligno, por lo que fue el ganador de esta ejecución.

Sin embargo, la ventaja fue pequeña. La segunda mejor configuración de validación cruzada fue lineal y quedó prácticamente empatada. Por eso mi conclusión no es que RBF siempre sea mejor, sino que obtuvo una mejora marginal en este protocolo. Antes de usarlo en un contexto real necesitaría validación externa, más datos y revisión médica. También existe riesgo en muestras ambiguas cercanas al margen, y todavía quedó un tumor maligno sin detectar. El modelo puede apoyar una decisión, pero no reemplazar un diagnóstico clínico.